# SAR Importance Ablation
---

In [1]:
# Change working directory to root

%cd ..

/mnt/c/Users/sebas/Documents/projects/multimodal-canopy


In [2]:
import random
import pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from models.dataset import CanopyDataset
from models.u_nets import DualEncoderUNet
from models.u_nets import SingleEncoderUNet
from torchinfo import summary
import matplotlib.pyplot as plt
from tqdm import tqdm

## Experimental Framework
---

In [3]:
SPLIT_SEED = 42

BATCH_SIZE = 16
EPOCHS = 200
LR = 1e-3
PATIENCE = 55

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

stack_dir = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/binary_stacks"
feature_range = (0,11)
target_idx = 11

dataset = CanopyDataset(stack_dir, feature_range, target_idx)

train_size = int(0.7 * len(dataset))
val_size = int(0.2 * len(dataset))
test_size = len(dataset) - train_size - val_size

split_generator = torch.Generator().manual_seed(SPLIT_SEED)

train_set, val_set, test_set = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=split_generator
)

cuda


In [4]:
# Define seeders

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        

def make_loaders(seed):

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=loader_generator
    )

    val_loader = DataLoader(
        val_set,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_set,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return train_loader, val_loader, test_loader

In [5]:
# Define metrics 

def masked_mse(pred, target, mask):
    valid = mask > 0
    if valid.sum() == 0:
        return (
            torch.tensor(0.0, device=pred.device),
            torch.tensor(0.0, device=pred.device),
            torch.tensor(0.0, device=pred.device)
        )
    
    sq_err = (pred - target)**2

    return (
        sq_err[valid].mean(),
        sq_err[valid].sum(),
        valid.sum()
    )

def masked_mae(pred, target, mask):
    valid = mask > 0
    if valid.sum() == 0:
        return (
            torch.tensor(0.0, device=pred.device),
            torch.tensor(0.0, device=pred.device),
            torch.tensor(0.0, device=pred.device)
        )

    abs_err = torch.abs(pred - target)

    return (
        abs_err[valid].mean(),
        abs_err[valid].sum(),
        valid.sum()
    )

def masked_me(pred, target, mask):
    valid = mask > 0
    if valid.sum() == 0:
        return (
            torch.tensor(0.0, device=pred.device),
            torch.tensor(0.0, device=pred.device),
            torch.tensor(0.0, device=pred.device)
        )

    err = pred - target

    return (
        err[valid].mean(),
        err[valid].sum(),
        valid.sum()
    )

In [6]:
# Define model factory

def make_model(model_type):

    if model_type == "sar":

        model = SingleEncoderUNet(
            n_in_channels=2,
            n_out_channels=1
        )

    elif model_type == "opt":

        model = SingleEncoderUNet(
            n_in_channels=9,
            n_out_channels=1
        )

    elif model_type == "early":

        model = SingleEncoderUNet(
            n_in_channels=11,
            n_out_channels=1
        )

    elif model_type == "dual":

        model = DualEncoderUNet(
            n_sar_channels=2,
            n_opt_channels=9,
            n_out_channels=1
        )

    else:
        raise ValueError(
            f"Unknown model type: {model_type}"
        )

    return model.to(DEVICE)

In [7]:
# Define training function

def train_model(
        model,
        optimizer,
        train_loader,
        val_loader,
        mode: str,
        epochs,
        checkpoint_fp,
        resume,
        patience,
        min_delta,
    ):

    if resume:
        checkpoint = torch.load(checkpoint_fp, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])

    best_val_loss = np.inf
    train_losses = []
    val_losses = []

    for epoch in tqdm(range(epochs), desc="Training model"):
        # ==============================
        # Training
        # ==============================
        model.train()

        total_squared_error = 0.0
        total_valid_pixels = 0

        for X, y, mask in train_loader:
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)

            opt = X[:, :9]
            sar = X[:, 9:]

            optimizer.zero_grad()

            if mode == "dual":
                pred = model(sar, opt)
            elif mode == "sar":
                pred = model(sar)
            elif mode == "opt":
                pred = model(opt)
            elif mode == "early":
                pred = model(X)
            else:
                raise ValueError("Unrecognized model mode.")

            batch_loss, sq_err, n_valid = masked_mse(pred, y, mask)

            batch_loss.backward()
            optimizer.step()

            total_squared_error += sq_err.item()
            total_valid_pixels += n_valid.item()

        train_loss = total_squared_error / total_valid_pixels


        # ==============================
        # Validation
        # ==============================
        model.eval()

        total_squared_error = 0.0
        total_valid_pixels = 0

        with torch.no_grad():
            for X, y, mask in val_loader:
                X = X.to(DEVICE)
                y = y.to(DEVICE)
                mask = mask.to(DEVICE)

                opt = X[:, :9]
                sar = X[:, 9:]

                if mode == "dual":
                    pred = model(sar, opt)
                elif mode == "sar":
                    pred = model(sar)
                elif mode == "opt":
                    pred = model(opt)
                elif mode == "early":
                    pred = model(X)
                else:
                    raise ValueError("Unrecognized model mode.")

                _, sq_err, n_valid = masked_mse(pred, y, mask)

                total_squared_error += sq_err.item()
                total_valid_pixels += n_valid.item()

        val_loss = total_squared_error / total_valid_pixels

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # ==============================
        # Checkpoint
        # ==============================
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            patience_counter = 0

            torch.save(
                {
                    "epoch": epoch+1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                    "train_loss": train_loss,
                },
                checkpoint_fp
            )
        else:
            patience_counter += 1

        # ==============================
        # Early stopping
        # ==============================
        if patience_counter >= patience:
            print(
                f"\nEarly stopping at epoch {epoch+  1}."
            )
            print( 
                f"Best validation loss: {best_val_loss:.4f} "
                f"at epoch {best_epoch}."
            )

            break

    return train_losses, val_losses, best_epoch

In [8]:
# Define test function

def test_model(
    model,
    test_loader,
    checkpoint_fp,
    mode
):
    checkpoint = torch.load(
        checkpoint_fp,
        map_location=DEVICE
    )
    model.load_state_dict(checkpoint["model_state_dict"])

    model.eval()

    total_squared_error = 0.0
    total_abs_error     = 0.0
    total_error         = 0.0
    total_valid_pixels  = 0.0

    with torch.no_grad():

        for X, y, mask in test_loader:

            X = X.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)

            opt = X[:, :9]
            sar = X[:, 9:]

            if mode == "dual":
                pred = model(sar, opt)
            elif mode == "sar":
                pred = model(sar)
            elif mode == "opt":
                pred = model(opt)
            elif mode == "early":
                pred = model(X)
            else:
                raise ValueError(f"Unrecognized model mode: '{mode}'")
            
            _, sq_err, n_valid = masked_mse(pred, y, mask)
            _, abs_error, _    = masked_mae(pred, y, mask) 
            _, error, _        = masked_me(pred, y, mask)

            total_squared_error += sq_err.item()
            total_abs_error += abs_error.item()
            total_error += error.item()
            total_valid_pixels += n_valid.item()

    test_mse = total_squared_error / total_valid_pixels
    test_mae = total_abs_error / total_valid_pixels
    test_me = total_error / total_valid_pixels

    return {
        "mse": test_mse,
        "rmse": np.sqrt(test_mse),
        "mae": test_mae,
        "me": test_me
    }

## Plotting Functions
---

In [9]:
# Define training history plotting function

def plot_training_history(
        train_losses, 
        val_losses,
        best_epoch,
        model_name,
        fig_fp,
    ):
    epochs_completed = np.arange(1, len(train_losses) + 1)

    plt.figure(figsize=(10, 6))

    plt.plot( 
        epochs_completed,
        train_losses,
        label="Training MSE"
    )

    plt.plot(
        epochs_completed,
        val_losses,
        label="Validation MSE"
    )

    plt.scatter(
        best_epoch,
        min(val_losses),    # Best val loss
        s=60,
        zorder=5,
        label=f"Best validation epoch: {best_epoch}"
    )

    plt.axvline(
        best_epoch,
        linestyle="--",
        alpha=0.5
    )

    plt.xlabel("Epoch")
    plt.ylabel("Masked MSE")
    plt.title(f"{model_name} Training History")

    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(fig_fp)
    plt.show()

In [10]:
# Define spatial erorr mapping function

def map_error(
    model,
    checkpoint_fp,
    mode,
    model_name,
    fig_fp
):
    checkpoint = torch.load(checkpoint_fp, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])

    model.eval()

    n_samples = 5

    fig, ax = plt.subplots(
        n_samples,
        5,
        figsize=(20, 4*n_samples),
        layout="constrained" # Replaces tight_layout to handle colorbar spacing cleanly
    )

    sample_count = 0

    for X,y,mask in test_loader:

        X = X.to(DEVICE)
        y = y.to(DEVICE)
        mask = mask.to(DEVICE)

        with torch.no_grad():
            opt = X[:, :9]
            sar = X[:, 9:]

            if mode == "dual":
                pred = model(sar, opt)
            elif mode == "sar":
                pred = model(sar)
            elif mode == "opt":
                pred = model(opt)
            elif mode == "early":
                pred = model(X)
            else:
                raise ValueError("Unrecognized mode.")
            
        for j in range(X.shape[0]):

            if sample_count >= n_samples:
                break

            rgb = X[j,:3].cpu().numpy()
            true = y[j,0].cpu().numpy()
            prediction = pred[j,0].cpu().numpy()
            valid = mask[j,0].cpu().numpy()

            true_masked = np.where(valid,true,np.nan)
            pred_masked = np.where(valid,prediction,np.nan)

            error = pred_masked - true_masked

            # 1. Plot RGB Image
            ax[sample_count,0].imshow(
                np.transpose(rgb,(1,2,0))*100
            )
            ax[sample_count,0].set_title("RGB")
            ax[sample_count,0].axis("off")

            # 2. Plot True CHM with shared scale
            vmin_chm = min(np.nanmin(true_masked), np.nanmin(pred_masked))
            vmax_chm = max(np.nanmax(true_masked), np.nanmax(pred_masked))

            im_true = ax[sample_count,1].imshow(
                true_masked, cmap="viridis", vmin=vmin_chm, vmax=vmax_chm
            )
            ax[sample_count,1].set_title("True CHM")
            ax[sample_count,1].axis("off")

            # 3. Plot Pred CHM
            im_pred = ax[sample_count,2].imshow(
                pred_masked, cmap="viridis", vmin=vmin_chm, vmax=vmax_chm
            )
            ax[sample_count,2].set_title("Pred CHM")
            ax[sample_count,2].axis("off")
            
            # Add a shared colorbar for True and Pred subplots in this row
            fig.colorbar(im_pred, ax=ax[sample_count, 1:3], orientation="vertical", shrink=0.8)

            # 4. Plot Error Map & Error Colorbar
            # Center the error colormap at 0 using symmetric vmin/vmax
            max_err = np.nanmax(np.abs(error))
            im_err = ax[sample_count,3].imshow(
                error, cmap="coolwarm", vmin=-max_err, vmax=max_err
            )
            ax[sample_count,3].set_title("Error")
            ax[sample_count,3].axis("off")
            
            fig.colorbar(im_err, ax=ax[sample_count,3], shrink=0.8)

            # 5. Scatter Plot
            ax[sample_count,4].scatter(
                true_masked.flatten(),
                pred_masked.flatten(),
                s=1,
                alpha=0.3
            )
            ax[sample_count,4].set_title("True vs Pred")
            ax[sample_count,4].set_xlabel("True CHM")       
            ax[sample_count,4].set_ylabel("Predicted CHM")

            sample_count += 1

        if sample_count >= n_samples:
            break

    fig.suptitle(f"{model_name} Error Map", fontsize=16, fontweight="bold")
    plt.savefig(fig_fp)
    plt.show()


## Experiment Run
---

In [17]:
MODEL_TYPES = [
    # "sar",
    # "opt",
    "early",
    "dual"
]

SEEDS = [
    1, 
    2,
    3,
    4,
    5
]

In [ ]:
histories = []
results = []

for model_type in MODEL_TYPES:

    print(f"\n{'='*60}")
    print(f"MODEL: {model_type.upper()}")
    print(f"{'='*60}")

    for seed in SEEDS:

        print(f"\nSeed: {seed}")

        # ========================
        # Set State
        # ========================

        # Set random state
        set_seed(seed)

        # Create loaders
        train_loader, val_loader, test_loader = make_loaders(seed)

        # Fresh model
        model = make_model(model_type)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LR
        )

        # Unique checkpoint
        checkpoint_fp = (
            f"models/weights/"
            f"best_{model_type}_seed{seed}.pth"
        )

        # ========================
        # Train
        # ========================

        train_losses, val_losses, best_epoch = train_model(
            model=model,
            optimizer=optimizer,
            train_loader=train_loader,
            val_loader=val_loader,
            mode=model_type,
            epochs=EPOCHS,
            checkpoint_fp=checkpoint_fp,
            resume=False,
            patience=PATIENCE,
            min_delta=0.001
        )

        # Save training history
        histories.append({
            "run_id": f"{model_type}_seed{seed}",
            "model": model_type,
            "seed": seed,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "best_epoch": best_epoch
        })
   
        # ========================
        # Test
        # ========================
        
        metrics = test_model(
            model=model,
            checkpoint_fp=checkpoint_fp,
            test_loader=test_loader,
            mode=model_type
        )

        metrics["model"] = model_type
        metrics["seed"] = seed  
        metrics["best_epoch"] = best_epoch 

        results.append(metrics)    
 


MODEL: EARLY

Seed: 1


Training model: 100%|██████████| 200/200 [1:06:54<00:00, 20.07s/it]



Seed: 2


Training model:  78%|███████▊  | 157/200 [52:38<17:55, 25.02s/it] 

In [ ]:
# Save experiment results

results_df = pd.DataFrame(results)
results_df.to_csv(
    "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/models/results/ablation_results.csv",
    index=False
)

# Save training histories
with open("/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/models/results/training_histories.pkl", "wb") as f:
    pickle.dump(histories, f)

In [ ]:
# View, summarize save results

summary_df = (
    results_df
    .groupby("model")
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        me_mean=("me", "mean"),
        me_std=("me", "std")
    )
)

results_summary_fp = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/models/results/results_summary.csv"
summary_df.to_csv(results_summary_fp)

summary_df